# Environment Setup

Import required libraries and initialize project configuration.

In [1]:
# Import libraries and configure project settings

import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import boto3
import sagemaker
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

PROJECT_PREFIX = "loanwise"

MODEL_FILE = "best_model.joblib"

sess = sagemaker.Session()

bucket = sess.default_bucket()

print(f"SageMaker Version : {sagemaker.__version__}")
print(f"Bucket            : {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker Version : 2.224.4
Bucket            : sagemaker-us-east-1-612256167011


# Load Model Artifacts

Load the best model generated during Notebook 2.

In [3]:
# Load best model artifact

model = joblib.load(
    MODEL_FILE
)

print(type(model))
print("Model loaded successfully.")

<class 'xgboost.sklearn.XGBClassifier'>
Model loaded successfully.


# Load Inference Dataset

Load the processed testing dataset and create a smaller inference sample.

In [4]:
# Load inference dataset

test_df = pd.read_csv(
    "processed_test.csv"
)

inference_df = test_df.sample(
    n=500,
    random_state=42
).reset_index(drop=True)

print(f"Inference Dataset Shape : {inference_df.shape}")

display(inference_df.head())

Inference Dataset Shape : (500, 18)


,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,DAYS_BIRTH,DAYS_EMPLOYED,CNT_FAM_MEMBERS,REGION_POPULATION_RELATIVE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,CODE_GENDER_M,FLAG_OWN_CAR_Y,FLAG_OWN_REALTY_Y,event_time
0,219711,0,0,76500.0,180000.0,9000.0,180000.0,-9187,-725,1.0,0.020713,0.179129,0.555917,0.468660,0,0,1,2026-06-13T11:39:45Z
1,446884,1,0,270000.0,395640.0,36288.0,360000.0,-10262,-477,1.0,0.046220,0.204936,0.206908,0.468660,0,0,1,2026-06-13T11:39:45Z
2,321881,1,0,135000.0,270000.0,18265.5,270000.0,-18344,-1793,2.0,0.006305,0.440128,0.000889,0.170446,0,0,0,2026-06-13T11:39:45Z
3,407616,1,0,157500.0,202500.0,10125.0,202500.0,-18028,-998,2.0,0.015221,0.337588,0.268869,0.166406,0,1,1,2026-06-13T11:39:45Z
4,280559,0,0,274500.0,1506816.0,47443.5,1350000.0,-19984,-8438,1.0,0.006629,0.440128,0.687704,0.450747,0,0,1,2026-06-13T11:39:45Z


# Prepare Inference Data

Separate model features from target labels and save the inference dataset.

In [5]:
# Prepare inference features

y_true = inference_df["TARGET"]

X_inference = inference_df.drop(
    columns=[
        "TARGET",
        "event_time"
    ]
)

X_inference.to_csv(
    "inference_dataset.csv",
    index=False
)

print(f"Inference Features Shape : {X_inference.shape}")

Inference Features Shape : (500, 16)


# Generate Predictions

Generate predictions and prediction probabilities.

In [6]:
# Generate predictions

predictions = model.predict(
    X_inference
)

probabilities = model.predict_proba(
    X_inference
)[:, 1]

prediction_df = pd.DataFrame({
    "Actual": y_true,
    "Prediction": predictions,
    "Probability": probabilities
})

display(prediction_df.head())

,Actual,Prediction,Probability
0,0,1,0.637472
1,1,1,0.808633
2,1,1,0.941834
3,1,1,0.639328
4,0,0,0.207672


# Evaluate Inference Results

Evaluate model performance on the inference sample.

In [7]:
# Calculate inference metrics

metrics = {
    "Accuracy": accuracy_score(
        y_true,
        predictions
    ),
    "Precision": precision_score(
        y_true,
        predictions
    ),
    "Recall": recall_score(
        y_true,
        predictions
    ),
    "F1": f1_score(
        y_true,
        predictions
    ),
    "ROC_AUC": roc_auc_score(
        y_true,
        predictions
    )
}

display(
    pd.DataFrame([metrics])
)

print(
    confusion_matrix(
        y_true,
        predictions
    )
)

,Accuracy,Precision,Recall,F1,ROC_AUC
0,0.678,0.687243,0.662698,0.674747,0.678123


[[172  76]
 [ 85 167]]


# Create Prediction Reports

Generate prediction outputs and summary reports.

In [8]:
# Create prediction reports

prediction_df.to_csv(
    "predictions.csv",
    index=False
)

summary = {
    "records_scored": len(prediction_df),
    "accuracy": metrics["Accuracy"],
    "precision": metrics["Precision"],
    "recall": metrics["Recall"],
    "f1_score": metrics["F1"],
    "roc_auc": metrics["ROC_AUC"]
}

with open(
    "prediction_summary.json",
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

display(summary)

{'records_scored': 500,
 'accuracy': 0.678,
 'precision': 0.6872427983539094,
 'recall': 0.6626984126984127,
 'f1_score': 0.6747474747474748,
 'roc_auc': 0.6781233998975934}

# Upload Artifacts to Amazon S3

Upload inference artifacts for monitoring and reporting.

In [9]:
# Upload inference artifacts

inference_s3_uri = sess.upload_data(
    path="inference_dataset.csv",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/inference"
)

predictions_s3_uri = sess.upload_data(
    path="predictions.csv",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/predictions"
)

summary_s3_uri = sess.upload_data(
    path="prediction_summary.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/reports"
)

print(inference_s3_uri)
print(predictions_s3_uri)
print(summary_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/inference/inference_dataset.csv
s3://sagemaker-us-east-1-612256167011/loanwise/predictions/predictions.csv
s3://sagemaker-us-east-1-612256167011/loanwise/reports/prediction_summary.json


# Verify Generated Outputs

Review generated prediction artifacts and S3 locations.

In [10]:
# Verify generated outputs

print("Generated Local Artifacts")
print("-------------------------")
print("inference_dataset.csv")
print("predictions.csv")
print("prediction_summary.json")

print("\nAmazon S3 Artifacts")
print("-------------------")
print(inference_s3_uri)
print(predictions_s3_uri)
print(summary_s3_uri)

display(prediction_df.head())

Generated Local Artifacts
-------------------------
inference_dataset.csv
predictions.csv
prediction_summary.json

Amazon S3 Artifacts
-------------------
s3://sagemaker-us-east-1-612256167011/loanwise/inference/inference_dataset.csv
s3://sagemaker-us-east-1-612256167011/loanwise/predictions/predictions.csv
s3://sagemaker-us-east-1-612256167011/loanwise/reports/prediction_summary.json


,Actual,Prediction,Probability
0,0,1,0.637472
1,1,1,0.808633
2,1,1,0.941834
3,1,1,0.639328
4,0,0,0.207672


# Summary and Next Steps

Notebook 4 successfully generated predictions using the approved model and created inference artifacts for monitoring and reporting.

Outputs generated:

- inference_dataset.csv
- predictions.csv
- prediction_summary.json

These artifacts will be used in Notebook 5 to implement model monitoring, data monitoring, and infrastructure monitoring.

In [11]:
# Notebook completion

print("Notebook 4 completed successfully.")

Notebook 4 completed successfully.
